In [ ]:
import numpy as np 
import pandas as pd 
import os 
import sys 
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt 
%matplotlib notebook
sys.path.insert(1,'../../../../arimaclust/')
from arimaclust.models.centroid import ConsistentARMAClustering

# USD

In [ ]:
df_usd = pd.read_csv('data/final_usd.csv',index_col='Unnamed: 0')

In [ ]:
df_usd.drop(columns=['VEF','USD'],inplace=True)

In [ ]:
df_usd=df_usd-df_usd.mean()

In [ ]:
df_usd['ETB'].describe().round(3)

In [ ]:
scaler = StandardScaler()

In [ ]:
df_usd_scaled = pd.DataFrame(scaler.fit_transform(df_usd),columns=df_usd.columns)

In [ ]:
from sklearn.preprocessing import StandardScaler

X = []
for col in df_usd.columns:
    X.append(df_usd_scaled[col].rolling(3).mean().dropna())
    
model = ConsistentARMAClustering(k=15, tau=10, max_orders=(3, 3))
model.fit(X=X)

In [ ]:
from sklearn import manifold 

mds = manifold.MDS(
    n_components=2,
    max_iter=3000,
    eps=1e-9,
    random_state=42,
    dissimilarity="precomputed",
    n_jobs=-1,
    normalized_stress="auto",
)
pos = mds.fit(model.D).embedding_


nmds = manifold.MDS(
    n_components=2,
    metric=False,
    max_iter=3000,
    eps=1e-12,
    dissimilarity="precomputed",
    random_state=42,
    n_jobs=-1,
    n_init=1,
    normalized_stress="auto",
)
pos = nmds.fit_transform(model.D, init=pos)

In [ ]:
fig, ax = plt.subplots(figsize=(12,8))
ax.scatter(pos[:,0],pos[:,1],c=model.labels_)
for i in range(df_usd.columns.shape[0]):
    ax.annotate(df_usd.columns[i], (pos[i][0], pos[i][1]),fontsize=8)

In [ ]:
for i in range(15):
    print(i)
    df_usd[df_usd_scaled.columns[model.labels_==i]].rolling(4).mean().plot()
    plt.show()

In [ ]:
df_usd

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage,to_tree, fcluster
from scipy.spatial.distance import squareform

In [ ]:
dist_df = pd.DataFrame(data=model.D, columns=df_usd.columns)

In [ ]:
dist_matrix_flat = squareform(dist_df)

In [ ]:
Z = linkage(dist_matrix_flat, 'ward',metric='precomputed')

In [ ]:

fig = plt.figure(figsize=(25, 10))
dn = dendrogram(Z,labels=df_usd.columns)
plt.show()

In [ ]:
list(df_usd.columns).index('AMD')

In [ ]:
l = fcluster(Z, t=0.05, criterion='distance')

In [ ]:
cluster = df_usd.columns[l==l[list(df_usd.columns).index('BTN')]]

In [ ]:
df_usd[cluster].rolling(3).mean().plot()

In [ ]:
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.sparse import csr_matrix

In [ ]:
D_thresh = np.where(model.D>0.1,0,1)

In [ ]:
D_thresh

In [ ]:
X_ = csr_matrix(D_thresh)

In [ ]:
mst = minimum_spanning_tree(X_)

In [ ]:
mst = mst.toarray().astype(int)

In [ ]:
plt.matshow(mst)